# Corinthians e o rebaixamento — Dixon-Coles + Monte Carlo

**Pergunta:** qual a probabilidade de o Corinthians terminar o Brasileirão (Série A) no Z4?

**Pipeline:** coleta (football-data.co.uk) → tabela atual → Dixon-Coles com decaimento temporal → simulação Monte Carlo dos jogos restantes → gráficos → sensibilidade → backtest (Brier + calibração).

## 1. Setup

In [ ]:
# !pip install pandas numpy scipy plotly requests nbformat
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.optimize import minimize
from scipy.stats import poisson

warnings.filterwarnings("ignore")
pio.templates.default = "plotly_white"

: 

In [ ]:
TEAM = "Corinthians"
N_SIMS = 20_000
N_RELEGATED = 4
XI = 0.0019            # decaimento temporal por dia (meia-vida ~ 1 ano)
WINDOW_YEARS = 3       # janela de treino
MAXG = 10              # gols máximos na grade de placares
SEED = 42
DATA_URL = "https://www.football-data.co.uk/new/BRA.csv"
DATA_PATH = Path("data/BRA.csv")

C_HL, C_OTHER, C_Z4, C_OK = "#111827", "#CBD5E1", "#EF4444", "#10B981"

## 2. Coleta de dados

O CSV do football-data.co.uk traz resultados (e odds) do Brasileirão. Ele é baixado uma vez e fica em cache em `data/BRA.csv`; use `load_data(force=True)` para atualizar. Se o download falhar, baixe o arquivo manualmente e salve nesse caminho.

In [ ]:
def load_data(force=False):
    if force or not DATA_PATH.exists():
        DATA_PATH.parent.mkdir(exist_ok=True)
        r = requests.get(DATA_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        r.raise_for_status()
        DATA_PATH.write_bytes(r.content)
    raw = pd.read_csv(DATA_PATH, encoding="latin-1")
    df = raw.rename(columns={"Home": "home", "Away": "away", "HG": "hg", "AG": "ag", "Season": "season"})
    if "League" in df.columns:
        df = df[df["League"].astype(str).str.contains("Serie A", case=False, na=True)]
    df["date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    return df

df = load_data()
played = df.dropna(subset=["hg", "ag"]).copy()
played[["hg", "ag"]] = played[["hg", "ag"]].astype(int)

SEASON = int(played["season"].max())
cur = played[played["season"] == SEASON].copy()
TEAMS = sorted(set(cur.home) | set(cur.away))
assert TEAM in TEAMS, f"{TEAM} não encontrado. Opções: {TEAMS}"

print(f"{len(played)} jogos | temporada {SEASON}: {len(cur)} jogados, {len(TEAMS)} times")
print(f"Último jogo: {cur.date.max():%d/%m/%Y}")
cur.tail(5)[["date", "home", "hg", "ag", "away"]]

### Tabela atual e jogos restantes

A Série A é pontos corridos em turno e returno: cada par (mandante, visitante) acontece uma vez. Os jogos restantes são os pares que ainda não apareceram nos resultados.

In [ ]:
def build_table(m):
    h = m.assign(team=m.home, gf=m.hg, ga=m.ag)
    a = m.assign(team=m.away, gf=m.ag, ga=m.hg)
    t = pd.concat([h, a])
    t["pts"] = np.select([t.gf > t.ga, t.gf == t.ga], [3, 1], 0)
    t["v"] = (t.gf > t.ga).astype(int)
    g = t.groupby("team").agg(j=("pts", "size"), pts=("pts", "sum"), v=("v", "sum"),
                              gp=("gf", "sum"), gc=("ga", "sum"))
    g["sg"] = g.gp - g.gc
    g = g.sort_values(["pts", "v", "sg", "gp"], ascending=False)
    g.insert(0, "pos", range(1, len(g) + 1))
    return g


def remaining_fixtures(season_df, teams):
    done = set(zip(season_df.home, season_df.away))
    return pd.DataFrame([(h, a) for h in teams for a in teams if h != a and (h, a) not in done],
                        columns=["home", "away"])


table = build_table(cur).reindex(TEAMS).fillna(0).sort_values(["pts", "v", "sg", "gp"], ascending=False)
table["pos"] = range(1, len(table) + 1)
fixtures = remaining_fixtures(cur, TEAMS)
print(f"Jogos restantes: {len(fixtures)}")
table

## 3. Exploração visual

In [ ]:
# --- 3.1 Tabela atual ---
t = table.iloc[::-1]
colors = [C_HL if i == TEAM else (C_Z4 if p > len(table) - N_RELEGATED else C_OTHER)
          for i, p in zip(t.index, t.pos)]
fig = go.Figure(go.Bar(
    x=t.pts, y=t.index, orientation="h", marker_color=colors,
    text=[f"{int(p)} pts · {int(j)} j" for p, j in zip(t.pts, t.j)], textposition="outside",
    hovertemplate="%{y}<br>%{x} pts<extra></extra>"))
fig.update_layout(title=f"Tabela atual — Brasileirão {SEASON}", height=620,
                  xaxis_title="Pontos", margin=dict(l=10, r=40, t=60, b=10))
fig.show()

In [ ]:
# --- 3.2 Corrida de pontos por jogo disputado ---
long = pd.concat([
    cur.assign(team=cur.home, gf=cur.hg, ga=cur.ag),
    cur.assign(team=cur.away, gf=cur.ag, ga=cur.hg)]).sort_values("date")
long["pts"] = np.select([long.gf > long.ga, long.gf == long.ga], [3, 1], 0)
long["n"] = long.groupby("team").cumcount() + 1
long["cum"] = long.groupby("team")["pts"].cumsum()

fig = go.Figure()
for tm, g in long.groupby("team"):
    if tm == TEAM:
        continue
    fig.add_trace(go.Scatter(x=g.n, y=g.cum, mode="lines", name=tm, line=dict(color=C_OTHER, width=1),
                             hovertemplate=f"{tm}<br>jogo %{{x}}: %{{y}} pts<extra></extra>"))
g = long[long.team == TEAM]
fig.add_trace(go.Scatter(x=g.n, y=g.cum, mode="lines+markers", name=TEAM,
                         line=dict(color=C_HL, width=4), marker=dict(size=5)))
fig.update_layout(title="Pontos acumulados por jogo disputado", xaxis_title="Jogos", yaxis_title="Pontos",
                  height=480, showlegend=False)
fig.show()

In [ ]:
# --- 3.3 Gols por jogo vs Poisson | Resultado do mandante por temporada ---
fig = make_subplots(rows=1, cols=2, subplot_titles=("Gols por time por jogo vs Poisson", "Mandante / Empate / Visitante"))
goals = pd.concat([played.hg, played.ag])
vals = np.arange(0, 8)
obs = goals.value_counts(normalize=True).reindex(vals).fillna(0)
fig.add_trace(go.Bar(x=vals, y=obs, name="Observado", marker_color=C_HL), 1, 1)
fig.add_trace(go.Scatter(x=vals, y=poisson.pmf(vals, goals.mean()), name="Poisson", mode="lines+markers",
                         line=dict(color=C_Z4, width=3)), 1, 1)

res = played.assign(r=np.select([played.hg > played.ag, played.hg == played.ag], ["Mandante", "Empate"], "Visitante"))
share = res.groupby(["season", "r"]).size().unstack().pipe(lambda d: d.div(d.sum(axis=1), axis=0))
for col, c in zip(["Mandante", "Empate", "Visitante"], [C_HL, C_OTHER, C_Z4]):
    fig.add_trace(go.Bar(x=share.index, y=share[col], name=col, marker_color=c), 1, 2)
fig.update_layout(barmode="stack", height=420, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title_text="Gols", row=1, col=1)
fig.show()

## 4. Modelo Dixon-Coles

Gols ~ Poisson com correção de dependência (ρ) nos placares baixos:

- λ (mandante) = exp(γ + ataque_casa − defesa_fora)
- μ (visitante) = exp(ataque_fora − defesa_casa)

Cada jogo pesa `exp(−ξ · dias_atrás)`. Defesa maior = defesa melhor. Os parâmetros somam zero e há uma regularização L2 leve para estabilidade.

In [ ]:
def tau(x, y, lam, mu, rho):
    t = np.ones_like(lam)
    m00, m01, m10, m11 = (x == 0) & (y == 0), (x == 0) & (y == 1), (x == 1) & (y == 0), (x == 1) & (y == 1)
    t[m00] = 1 - lam[m00] * mu[m00] * rho
    t[m01] = 1 + lam[m01] * rho
    t[m10] = 1 + mu[m10] * rho
    t[m11] = 1 - rho
    return t


def fit_dc(matches, ref_date, xi=XI, reg=0.01):
    teams = sorted(set(matches.home) | set(matches.away))
    idx = {t: i for i, t in enumerate(teams)}
    n = len(teams)
    hi, ai = matches.home.map(idx).values, matches.away.map(idx).values
    x, y = matches.hg.values.astype(int), matches.ag.values.astype(int)
    w = np.exp(-xi * (ref_date - matches.date).dt.days.clip(lower=0).values)

    def unpack(p):
        atk = np.append(p[:n - 1], -p[:n - 1].sum())
        dfn = np.append(p[n - 1:2 * n - 2], -p[n - 1:2 * n - 2].sum())
        return atk, dfn, p[-2], p[-1]

    def nll(p):
        atk, dfn, gamma, rho = unpack(p)
        lam = np.exp(gamma + atk[hi] - dfn[ai])
        mu = np.exp(atk[ai] - dfn[hi])
        tv = tau(x, y, lam, mu, rho)
        if np.any(tv <= 0):
            return 1e10
        ll = w * (np.log(tv) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu))
        return -ll.sum() + reg * np.sum(p[:2 * n - 2] ** 2)

    p0 = np.zeros(2 * n)
    p0[-2], p0[-1] = 0.25, -0.05
    bounds = [(None, None)] * (2 * n - 1) + [(-0.3, 0.3)]
    res = minimize(nll, p0, method="L-BFGS-B", bounds=bounds)
    atk, dfn, gamma, rho = unpack(res.x)
    return dict(atk=pd.Series(atk, teams), dfn=pd.Series(dfn, teams), gamma=gamma, rho=rho, ok=res.success)


ref_date = cur.date.max()
train = played[(played.date <= ref_date) & (played.date > ref_date - pd.DateOffset(years=WINDOW_YEARS))]
params = fit_dc(train, ref_date)
print(f"Convergiu: {params['ok']} | mando (γ)={params['gamma']:.3f} | ρ={params['rho']:.3f} | jogos no treino: {len(train)}")

In [ ]:
# --- 4.1 Mapa de força: ataque x defesa ---
s = pd.DataFrame({"atk": params["atk"], "dfn": params["dfn"]}).reindex(TEAMS)
fig = go.Figure(go.Scatter(
    x=s.atk, y=s.dfn, mode="markers+text", text=s.index, textposition="top center",
    marker=dict(size=[18 if i == TEAM else 11 for i in s.index],
                color=[C_HL if i == TEAM else C_OTHER for i in s.index], line=dict(width=1, color="#475569")),
    hovertemplate="%{text}<br>ataque %{x:.2f}<br>defesa %{y:.2f}<extra></extra>"))
fig.add_hline(y=0, line_dash="dot", line_color="#94A3B8")
fig.add_vline(x=0, line_dash="dot", line_color="#94A3B8")
fig.update_layout(title="Força dos times (acima/direita = melhor)", xaxis_title="Ataque", yaxis_title="Defesa", height=560)
fig.show()

## 5. Simulação Monte Carlo

In [ ]:
G = MAXG + 1


def score_probs(lam, mu, rho):
    g = np.arange(G)
    P = poisson.pmf(g[None, :, None], lam[:, None, None]) * poisson.pmf(g[None, None, :], mu[:, None, None])
    P[:, 0, 0] *= 1 - lam * mu * rho
    P[:, 0, 1] *= 1 + lam * rho
    P[:, 1, 0] *= 1 + mu * rho
    P[:, 1, 1] *= 1 - rho
    return P / P.sum(axis=(1, 2), keepdims=True)


def simulate(params, fixtures, base, n_sims=N_SIMS, seed=SEED, adj=None):
    # adj: {time: delta} soma delta (escala log) ao ataque e à defesa do time
    rng = np.random.default_rng(seed)
    teams = list(base.index)
    T, M = len(teams), len(fixtures)
    tix = {t: i for i, t in enumerate(teams)}
    atk = params["atk"].reindex(teams).fillna(0).copy()
    dfn = params["dfn"].reindex(teams).fillna(0).copy()
    for tm, d in (adj or {}).items():
        atk[tm] += d
        dfn[tm] += d
    hi, ai = fixtures.home.map(tix).values, fixtures.away.map(tix).values
    lam = np.exp(params["gamma"] + atk.values[hi] - dfn.values[ai])
    mu = np.exp(atk.values[ai] - dfn.values[hi])

    cdf = score_probs(lam, mu, params["rho"]).reshape(M, -1).cumsum(axis=1)
    u = rng.random((n_sims, M))
    flat = np.empty((n_sims, M), dtype=np.int32)
    for j in range(M):
        flat[:, j] = np.searchsorted(cdf[j], u[:, j])
    flat = np.minimum(flat, G * G - 1)
    hg, ag = flat // G, flat % G

    H = np.zeros((M, T)); H[np.arange(M), hi] = 1
    A = np.zeros((M, T)); A[np.arange(M), ai] = 1
    hw, aw, dr = (hg > ag).astype(float), (ag > hg).astype(float), (hg == ag).astype(float)
    pts = base.pts.values + (3 * hw + dr) @ H + (3 * aw + dr) @ A
    wins = base.v.values + hw @ H + aw @ A
    gf = base.gp.values + hg @ H + ag @ A
    gd = base.sg.values + (hg - ag) @ H + (ag - hg) @ A

    key = pts * 1e8 + wins * 1e6 + (gd + 300) * 1e3 + gf + rng.random((n_sims, T)) * 0.5
    order = np.argsort(-key, axis=1)
    pos = np.empty((n_sims, T), dtype=np.int16)
    np.put_along_axis(pos, order, np.broadcast_to(np.arange(1, T + 1, dtype=np.int16), (n_sims, T)), axis=1)
    return dict(teams=teams, pos=pos, pts=pts)


def summarize(sim):
    pos, T = sim["pos"], sim["pos"].shape[1]
    pp = pd.DataFrame(np.stack([(pos == k).mean(axis=0) for k in range(1, T + 1)], axis=1),
                      index=sim["teams"], columns=range(1, T + 1))
    return pp, pp.loc[:, T - N_RELEGATED + 1:].sum(axis=1)


sim = simulate(params, fixtures, table)
pos_probs, p_rel = summarize(sim)
i = sim["teams"].index(TEAM)
print(f"P({TEAM} no Z4) = {p_rel[TEAM]:.1%} | posição esperada = {sim['pos'][:, i].mean():.1f} | pontos esperados = {sim['pts'][:, i].mean():.1f}")

## 6. Resultados

In [ ]:
# --- 6.1 Probabilidade de rebaixamento por time ---
r = p_rel.sort_values()
fig = go.Figure(go.Bar(
    x=r.values, y=r.index, orientation="h",
    marker_color=[C_HL if t == TEAM else (C_Z4 if v > 0.5 else C_OTHER) for t, v in r.items()],
    text=[f"{v:.1%}" for v in r.values], textposition="outside",
    hovertemplate="%{y}: %{x:.1%}<extra></extra>"))
fig.update_layout(title=f"Probabilidade de rebaixamento ({N_SIMS:,} simulações)", height=620,
                  xaxis=dict(tickformat=".0%", range=[0, min(1.1, r.max() * 1.2 + 0.05)]),
                  margin=dict(l=10, r=40, t=60, b=10))
fig.show()

In [ ]:
# --- 6.2 Heatmap: probabilidade de cada posição final ---
order_t = pos_probs.mul(pos_probs.columns).sum(axis=1).sort_values().index
fig = go.Figure(go.Heatmap(
    z=pos_probs.loc[order_t].values, x=list(pos_probs.columns), y=list(order_t), colorscale="Blues",
    hovertemplate="%{y} em %{x}º: %{z:.1%}<extra></extra>", colorbar=dict(tickformat=".0%")))
fig.add_vrect(x0=len(TEAMS) - N_RELEGATED + 0.5, x1=len(TEAMS) + 0.5, fillcolor=C_Z4, opacity=0.12, line_width=0)
fig.update_layout(title="Distribuição da posição final", height=640, yaxis=dict(autorange="reversed"),
                  xaxis=dict(title="Posição final", dtick=1))
fig.show()

In [ ]:
# --- 6.3 Corinthians: posição final e pontos finais ---
pos_c, pts_c = sim["pos"][:, i], sim["pts"][:, i]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Posição final", "Pontos finais"))
pc = pos_probs.loc[TEAM]
fig.add_trace(go.Bar(x=pc.index, y=pc.values,
                     marker_color=[C_Z4 if p > len(TEAMS) - N_RELEGATED else C_HL for p in pc.index],
                     hovertemplate="%{x}º: %{y:.1%}<extra></extra>"), 1, 1)
fig.add_trace(go.Histogram(x=pts_c, xbins=dict(size=1), marker_color=C_HL, histnorm="probability",
                           hovertemplate="%{x} pts: %{y:.1%}<extra></extra>"), 1, 2)
fig.add_vline(x=45, line_dash="dash", line_color=C_Z4, annotation_text="corte histórico ~45", row=1, col=2)
fig.add_vline(x=np.median(pts_c), line_dash="dot", line_color=C_OK, annotation_text="mediana", row=1, col=2)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(title=f"{TEAM}: P(Z4) = {p_rel[TEAM]:.1%}", height=430, showlegend=False)
fig.update_xaxes(title_text="Posição", dtick=1, row=1, col=1)
fig.show()

## 7. Sensibilidade

E se a força real do time for diferente do que o modelo estima (técnico novo, lesões, reforços)? `delta` é um ajuste em escala log somado ao ataque e à defesa do time: positivo melhora, negativo piora.

In [ ]:
deltas = np.linspace(-0.3, 0.3, 9)
sens = []
for d in deltas:
    s_ = simulate(params, fixtures, table, n_sims=5_000, adj={TEAM: d})
    sens.append(summarize(s_)[1][TEAM])

fig = go.Figure(go.Scatter(x=deltas, y=sens, mode="lines+markers", line=dict(color=C_HL, width=3),
                           hovertemplate="delta %{x:.2f}<br>P(Z4) %{y:.1%}<extra></extra>"))
fig.add_vline(x=0, line_dash="dot", line_color="#94A3B8", annotation_text="modelo")
fig.update_layout(title=f"Sensibilidade: P(Z4) de {TEAM} vs ajuste de força", xaxis_title="Ajuste de força (log)",
                  yaxis=dict(title="P(Z4)", tickformat=".0%"), height=420)
fig.show()

## 8. Backtest (Brier + calibração)

Para cada temporada completa: congela o campeonato na metade, ajusta o modelo só com o que existia até ali, simula o resto e compara a P(Z4) de cada time com o rebaixamento real. Referência ingênua: dar 20% (4/20) para todo mundo, o que dá Brier = 0,16.

In [ ]:
def backtest(seasons, cutoff=0.5, n_sims=3_000):
    out = []
    for s in seasons:
        sdf = played[played.season == s].sort_values("date")
        if len(sdf) < 300:      # temporada incompleta
            continue
        past = sdf.iloc[: int(len(sdf) * cutoff)]
        ref = past.date.max()
        tr = played[(played.date <= ref) & (played.date > ref - pd.DateOffset(years=WINDOW_YEARS))]
        prm = fit_dc(tr, ref)
        teams_s = sorted(set(sdf.home) | set(sdf.away))
        base = build_table(past).reindex(teams_s).fillna(0)
        sm = simulate(prm, remaining_fixtures(past, teams_s), base, n_sims=n_sims)
        _, pr = summarize(sm)
        real = set(build_table(sdf).index[-N_RELEGATED:])
        out += [dict(season=s, team=t, p=pr[t], y=int(t in real)) for t in teams_s]
    return pd.DataFrame(out)


bt_seasons = [s for s in sorted(played.season.unique()) if s < SEASON][-8:]
bt = backtest(bt_seasons)
brier = ((bt.p - bt.y) ** 2).mean()
print(f"Temporadas: {sorted(bt.season.unique())}")
print(f"Brier modelo = {brier:.4f} | baseline ingênuo (20%) = {((0.2 - bt.y) ** 2).mean():.4f}")

In [ ]:
# --- Curva de calibração ---
bt["bin"] = pd.cut(bt.p, [0, .05, .1, .2, .35, .5, .7, 1.0], include_lowest=True)
cal = bt.groupby("bin").agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size")).dropna()
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(dash="dash", color="#94A3B8"), name="Calibração perfeita"))
fig.add_trace(go.Scatter(x=cal.p, y=cal.y, mode="lines+markers", name="Modelo",
                         marker=dict(size=np.sqrt(cal.n) * 3 + 6, color=C_HL), line=dict(color=C_HL),
                         customdata=cal.n, hovertemplate="previsto %{x:.1%}<br>observado %{y:.1%}<br>n=%{customdata}<extra></extra>"))
fig.update_layout(title=f"Calibração do P(Z4) — Brier {brier:.3f}", xaxis=dict(title="Previsto", tickformat=".0%"),
                  yaxis=dict(title="Observado", tickformat=".0%"), height=480, width=560)
fig.show()

## 9. Próximos passos

- **Benchmark com odds:** as colunas `AvgH/AvgD/AvgA` do CSV dão probabilidades implícitas por jogo; compare o Brier/log-loss dos resultados 1x2 do Dixon-Coles contra elas.
- **Extensão HMM / força dinâmica:** substitua `fit_dc` por uma função que devolva ataque/defesa variando no tempo (ou por estado latente) e mantenha `simulate` igual, trocando só o cálculo de `lam` e `mu`. Compare no mesmo `backtest`.
- **Ajustes:** `XI` e `WINDOW_YEARS` podem ser escolhidos por validação cruzada temporal.